<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/English_to_Ukrainian_Colab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Fast Google-Powered Word Translator (No API Key Required)
# @markdown This version is lightweight, fast, and uses Google's translation engine for free.

import ipywidgets as widgets
from IPython.display import display, Javascript
import io
import os
import requests
import json
import base64
import time

# --- INSTALL DEPENDENCIES ---
try:
    from docx import Document
except ImportError:
    os.system('pip install python-docx -q')
    from docx import Document

def google_translate(text, src='en', tgt='uk'):
    """Uses the free Google Translate Ajax endpoint (No API Key)."""
    try:
        url = "https://translate.googleapis.com/translate_a/single"
        params = {
            "client": "gtx",
            "sl": src,
            "tl": tgt,
            "dt": "t",
            "q": text
        }
        # Add a timeout and retry logic for reliability
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            # The API returns a nested list; we extract the translated strings
            result_parts = response.json()[0]
            translated_text = "".join([part[0] for part in result_parts if part[0]])
            return translated_text
        return None
    except Exception as e:
        print(f"Translation segment error: {e}")
        return None

def translate_text(btn):
    text = ""
    file_suffix = "en_to_ua"

    # Reset UI state
    output_text.value = ""
    output_label.value = "⏳ Initializing..."
    translate_btn.disabled = True
    translate_btn.description = "Processing..."

    # 1. Handle File Upload or Manual Text
    if file_upload.value:
        try:
            # FileUpload returns a dictionary in Colab
            file_info = list(file_upload.value.values())[0]
            file_name = file_info['metadata']['name']
            content = file_info['content']

            if file_name.endswith('.docx'):
                doc = Document(io.BytesIO(content))
                text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
            else:
                text = content.decode('utf-8').strip()
        except Exception as e:
            output_label.value = f"❌ File Error: {str(e)}"
            translate_btn.disabled = False
            translate_btn.description = "Start Fast Translation"
            return
    else:
        text = input_text.value.strip()

    if not text:
        output_label.value = "⚠️ Please upload a file or enter text."
        translate_btn.disabled = False
        translate_btn.description = "Start Fast Translation"
        return

    # 2. Process Translation
    try:
        direction = direction_dropdown.value
        src_code = 'en' if direction == "English to Ukrainian" else 'uk'
        tgt_code = 'uk' if direction == "English to Ukrainian" else 'en'
        file_suffix = "en_to_ua" if direction == "English to Ukrainian" else "ua_to_en"

        output_label.value = "⚙️ Translating via Google (No API)..."

        # Split text into paragraphs to handle large documents and API limits
        paragraphs = text.split('\n')
        translated_paragraphs = []

        for p in paragraphs:
            if p.strip():
                # Perform translation for each paragraph
                result = google_translate(p, src_code, tgt_code)
                if result:
                    translated_paragraphs.append(result)
                else:
                    translated_paragraphs.append("[Translation Failed]")
            else:
                translated_paragraphs.append("")

        final_result = "\n".join(translated_paragraphs)
        output_text.value = final_result

        # 3. Create .docx File
        output_label.value = "📄 Building Document..."
        out_doc = Document()
        for line in translated_paragraphs:
            if line.strip():
                out_doc.add_paragraph(line)

        doc_io = io.BytesIO()
        out_doc.save(doc_io)
        doc_io.seek(0)

        # 4. Trigger Automatic Download via Javascript
        b64 = base64.b64encode(doc_io.read()).decode()
        filename = f"translated_{file_suffix}.docx"
        js_download = f"""
        var element = document.createElement('a');
        element.setAttribute('href', 'data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,' + '{b64}');
        element.setAttribute('download', '{filename}');
        element.style.display = 'none';
        document.body.appendChild(element);
        element.click();
        document.body.removeChild(element);
        """
        display(Javascript(js_download))
        output_label.value = "✅ Success! Document downloaded."

    except Exception as e:
        output_label.value = f"❌ Error: {str(e)}"

    translate_btn.disabled = False
    translate_btn.description = "Start Fast Translation"

# --- UI SETUP ---
direction_dropdown = widgets.Dropdown(
    options=['English to Ukrainian', 'Ukrainian to English'],
    value='English to Ukrainian',
    description='Direction:',
    layout=widgets.Layout(width='300px')
)

file_upload = widgets.FileUpload(
    accept='.docx,.txt',
    multiple=False,
    description='Upload Document',
    button_style='info'
)

input_text = widgets.Textarea(
    placeholder='Paste text here if not uploading a file...',
    layout=widgets.Layout(width='95%', height='180px')
)

output_text = widgets.Textarea(
    layout=widgets.Layout(width='95%', height='180px'),
    disabled=True,
    placeholder='Translation preview will appear here...'
)

translate_btn = widgets.Button(
    description='Start Fast Translation',
    button_style='success',
    layout=widgets.Layout(width='300px', height='45px', margin='10px 0 10px 0')
)

output_label = widgets.Label(value="Status: Ready (No API needed)")
translate_btn.on_click(translate_text)

# Assemble Layout
ui = widgets.VBox([
    widgets.HTML("<h2 style='color: #1a73e8;'>Fast Bi-Directional Doc Translator</h2>"),
    widgets.HTML("<b>Step 1:</b> Select direction and upload a .docx or .txt file."),
    widgets.HBox([direction_dropdown, file_upload]),
    widgets.HTML("<br><b>Step 2:</b> Or paste text manually below."),
    input_text,
    translate_btn,
    output_label,
    widgets.HTML("<b>Preview:</b>"),
    output_text
])

display(ui)

<IPython.core.display.Javascript object>